# RAG baseline vs cross-encoder re-ranking

## 1. Config
All settings live in one cell. Set the environment variable `DOC` to `nfl` or `nba` to pick the rulebook. Both pipelines share the same chunks, retrieval depth (`TOP_K`) and prompt.

In [1]:
import os, re, time, random, statistics, urllib.request
import numpy as np, pandas as pd, torch, faiss, pymupdf
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

DOC = os.getenv("DOC", "nfl")   # "nfl" or "nba"
PDF_URLS = {
    "nfl": "https://media.nfl.com/content/dam/communications/Official%20Playing%20Rules/2026%20Official%20Playing%20Rules.pdf",
    "nba": "https://cdn.nba.com/manage/2026/01/Official-2025-26-NBA-Playing-Rules.pdf",
}
DOC_NAMES = {"nfl": "NFL 2026 Official Playing Rules", "nba": "NBA 2025-26 Official Playing Rules"}

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # bi-encoder (stage 1)
RERANK_MODEL = "cross-encoder/ettin-reranker-1b-v1"       # cross-encoder (stage 2)
LLM_MODEL    = "gemini-3.5-flash-lite"                    # Gemini API (needs GOOGLE_API_KEY in .env)
LLM_URL      = "https://generativelanguage.googleapis.com/v1beta/openai/"   # Gemini's OpenAI-compatible endpoint

TOP_K, TOP_N = 10, 3             # retrieve 10, send 3 to the LLM
CHUNK_WORDS, OVERLAP = 120, 30
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
SEED = 42

# Seed every RNG we use so reruns are repeatable
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Load GOOGLE_API_KEY from .env if the file exists (the key is never printed)
if os.path.exists(".env"):
    for line in open(".env"):
        k, _, v = line.strip().partition("=")
        if k and not k.startswith("#") and v:
            os.environ.setdefault(k, v.strip().strip("'").strip('"'))
print(DOC_NAMES[DOC], "| device:", DEVICE)

/Users/va/LMA A1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NBA 2025-26 Official Playing Rules | device: mps


## 2. Load PDF
Download the rulebook to `data/` if it is not already there, then extract the text of each page with PyMuPDF and collapse all whitespace runs to single spaces.

In [2]:
os.makedirs("data", exist_ok=True)
pdf_path = f"data/{DOC}.pdf"

# Download once; a browser-like User-Agent avoids 403s from the CDNs
if not os.path.exists(pdf_path):
    req = urllib.request.Request(PDF_URLS[DOC], headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(pdf_path, "wb") as f:
        f.write(resp.read())

# One cleaned string per page (list index 0 = page 1)
with pymupdf.open(pdf_path) as pdf:
    pages = [re.sub(r"\s+", " ", page.get_text()).strip() for page in pdf]
print(f"{len(pages)} pages, {sum(len(p.split()) for p in pages):,} words")

76 pages, 36,860 words


## 3. Chunk
Split each page into fixed windows of `CHUNK_WORDS` words with `OVERLAP` words shared between neighbours; windows never cross a page boundary, so every chunk has one page number. The sanity cell checks how many chunks exceed MiniLM's 256-token input limit (longer text is silently truncated).

In [3]:
chunks = []
step = CHUNK_WORDS - OVERLAP
for page_no, text in enumerate(pages, start=1):
    words = text.split()
    for start in range(0, len(words), step):
        window = words[start:start + CHUNK_WORDS]
        chunks.append({"id": len(chunks), "page": page_no, "text": " ".join(window)})
        if start + CHUNK_WORDS >= len(words):   # window reached page end, stop (avoids a tiny duplicate tail chunk)
            break
print("chunks:", len(chunks))

chunks: 422


In [4]:
# Sanity check: MiniLM truncates input at 256 tokens, so few chunks should be longer than that
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
n_tokens = [len(ids) for ids in tok([c["text"] for c in chunks], add_special_tokens=True)["input_ids"]]
pct_trunc = 100 * np.mean(np.array(n_tokens) > 256)
print(f"chunks: {len(chunks)} | max tokens: {max(n_tokens)} | median: {int(np.median(n_tokens))} | over 256 tokens: {pct_trunc:.2f}%")

chunks: 422 | max tokens: 204 | median: 147 | over 256 tokens: 0.00%


## 4. Index
Embed every chunk with the MiniLM bi-encoder and store the vectors in an exact FAISS flat index. The vectors are L2-normalised, so inner product equals cosine similarity. The Ettin cross-encoder is also loaded in this cell, but it is not part of the index. A cross-encoder needs the query and the chunk together, so it can only score (query, chunk) pairs at query time, inside `rerank`.

In [5]:
bi = SentenceTransformer(EMBED_MODEL, device=DEVICE)
texts = [c["text"] for c in chunks]
emb = bi.encode(texts, normalize_embeddings=True, batch_size=64)   # shape (n_chunks, 384)

index = faiss.IndexFlatIP(emb.shape[1])   # exact inner-product search
index.add(emb.astype("float32"))
print("indexed vectors:", index.ntotal)

ce = CrossEncoder(RERANK_MODEL, device=DEVICE)   # scores (query, chunk) pairs jointly

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12513.42it/s]

indexed vectors: 422


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 10296.19it/s]

## 5. Core functions
`retrieve` does the FAISS top-k search, `rerank` re-scores those candidates with the cross-encoder, and `generate` asks Gemini using one shared prompt. `run` executes both pipelines on a query: Pipeline 1 sends the bi-encoder top-3 to the LLM, Pipeline 2 sends the cross-encoder top-3 of the same 10 candidates.

In [6]:
PROMPT = """You are answering questions about the {doc_name}.
Use ONLY the excerpts below. If they do not contain the answer, say "Not covered in the provided excerpts."
Cite the page number(s) you used.

Excerpts:
{context}

Question: {question}
Answer:"""

assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env (see README)"
llm = OpenAI(base_url=LLM_URL, api_key=os.environ["GOOGLE_API_KEY"], max_retries=5)   # retries also cover rate limits


def retrieve(q, k=TOP_K):
    """Bi-encoder stage: embed the query, return top-k chunk ids and cosine scores."""
    qv = bi.encode([q], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    return ids[0].tolist(), scores[0].tolist()


def rerank(q, ids):
    """Cross-encoder stage: score each (query, chunk) pair, return ids sorted by score (high to low)."""
    scores = ce.predict([(q, chunks[i]["text"]) for i in ids])
    order = np.argsort(-scores)
    return [ids[j] for j in order], [float(scores[j]) for j in order]


def build_prompt(q, ids):
    """Fill the shared prompt with the selected chunks, each tagged with its page number."""
    context = "\n\n".join(f"[p. {chunks[i]['page']}] {chunks[i]['text']}" for i in ids)
    return PROMPT.format(doc_name=DOC_NAMES[DOC], context=context, question=q)


def generate(q, ids):
    """Ask Gemini with the shared prompt (temperature 0)."""
    resp = llm.chat.completions.create(
        model=LLM_MODEL, messages=[{"role": "user", "content": build_prompt(q, ids)}],
        temperature=0, max_tokens=2048,
    )
    return (resp.choices[0].message.content or "").strip()


def run(q):
    """Run both pipelines on one query and time every stage (seconds)."""
    out = {}
    t = time.perf_counter()
    out["base_ids"], out["base_scores"] = retrieve(q)
    out["t_retrieve"] = time.perf_counter() - t

    t = time.perf_counter()
    out["rr_ids"], out["rr_scores"] = rerank(q, out["base_ids"])
    out["t_rerank"] = time.perf_counter() - t

    t = time.perf_counter()
    out["answer1"] = generate(q, out["base_ids"][:TOP_N])   # Pipeline 1: baseline top-3
    out["t_llm1"] = time.perf_counter() - t

    t = time.perf_counter()
    out["answer2"] = generate(q, out["rr_ids"][:TOP_N])     # Pipeline 2: re-ranked top-3
    out["t_llm2"] = time.perf_counter() - t
    return out

## 6. Probe helper
`show` prints the retrieved top-10 for a query with bi-encoder and cross-encoder ranks and scores, without calling the LLM. A chunk counts as gold if any gold snippet appears in its text (case-insensitive, whitespace-normalised); `gold_ranks` returns the best gold rank under each ordering.

In [7]:
def norm(s):
    """Lowercase and collapse whitespace so snippet matching ignores formatting."""
    return re.sub(r"\s+", " ", s).lower().strip()


def is_gold(chunk_id, gold):
    text = norm(chunks[chunk_id]["text"])
    return any(norm(g) in text for g in (gold or []))


def show(q, gold=None):
    """Top-10 table: bi-encoder vs cross-encoder rank/score, with a gold flag. No LLM call."""
    base_ids, base_scores = retrieve(q)
    rr_ids, rr_scores = rerank(q, base_ids)
    ce_rank = {cid: r for r, cid in enumerate(rr_ids, start=1)}
    ce_score = dict(zip(rr_ids, rr_scores))
    rows = [{
        "chunk": cid,
        "page": chunks[cid]["page"],
        "bi_rank": r,
        "bi_score": round(s, 3),
        "ce_rank": ce_rank[cid],
        "ce_score": round(ce_score[cid], 3),
        "text": chunks[cid]["text"][:90],
        "gold": is_gold(cid, gold),
    } for r, (cid, s) in enumerate(zip(base_ids, base_scores), start=1)]
    return pd.DataFrame(rows)


def gold_ranks(q, gold):
    """Best (lowest) rank of any gold chunk in the bi-encoder and cross-encoder orderings; None if not in the top-10."""
    base_ids, _ = retrieve(q)
    rr_ids, _ = rerank(q, base_ids)
    first = lambda ids: next((r for r, cid in enumerate(ids, start=1) if is_gold(cid, gold)), None)
    return first(base_ids), first(rr_ids)

## 7. Failure set
Each query below has a gold snippet copied from the rulebook text. `show` prints the top-10 for every query, `run` executes both pipelines once per query, and the tables report where the gold chunk sits under each ranking and what each pipeline answered.

In [8]:
# Queries with gold snippets (substrings of the real chunk text). "control" and "CE did not help" rows are kept on purpose.
FAILURE_SETS = {
    "nfl": [
        {"query": "Can a coach challenge a play that begins after the two-minute warning?", "category": "exception",
         "gold": ["Only the Replay Official or the Senior Vice President of Officiating or his or her designee may initiate a review"]},
        # Distractor: the baseline top 3 are approved-ruling (A.R.) casebook chunks (p. 142, p. 194) that share "fumble" and
        # "two-minute warning" with the query; the rule itself (Article 6, p. 42) is at baseline rank 10.
        {"query": "Who is allowed to advance a fumble after the two-minute warning?", "category": "keyword trap / distractor",
         "gold": ["The player who fumbled is the only player of his team who is permitted to recover and advance the ball"]},
        {"query": "How many challenges does a team get?", "category": "buried",
         "gold": ["A team will be permitted a third challenge if it is successful on at least one of its challenges"]},
        {"query": "How long is the break between overtime periods?", "category": "buried",
         "gold": ["Between each overtime period, there shall be a two-minute intermission"]},
        {"query": "Can a team use two charged timeouts back to back?", "category": "negation",
         "gold": ["a second charged team timeout by either team during the same dead ball period is prohibited"]},
        {"query": "Can a kicker wear a detachable kicking toe?", "category": "negation",
         "gold": ["A detachable kicking toe is prohibited"]},
        {"query": "Which offensive players are not allowed to catch a forward pass?", "category": "control (already top 3)",
         "gold": ["are ineligible to catch a legal or illegal forward pass thrown from behind the line of scrimmage"]},
    ],
    "nba": [
        {"query": "Can a player be granted a timeout while airborne?", "category": "negation",
         "gold": ["A player shall not be granted any timeout if both of his feet are in the air"]},
        {"query": "Does the shot clock reset to 14 or 24 after a foul?", "category": "keyword trap",
         "gold": ["or stay the same or reset to 14, whichever is greater"]},
        {"query": "Can a coach walk past half court?", "category": "negation",
         "gold": ["A coach is not permitted to cross the midcourt line"]},
        {"query": "Can a substitute replace the player shooting free throws?", "category": "exception",
         "gold": ["shall not replace a free throw shooter or a player involved in a jump ball unless dictated to do so by an injury or ejection"]},
        {"query": "What are the mandatory timeouts in each period?", "category": "buried",
         "gold": ["There must be two mandatory timeouts in each period"]},
        {"query": "What makes a shot attempt legal for the shot clock?", "category": "buried",
         "gold": ["the ball must make contact with the basket ring"]},
        {"query": "Does an injured player's free throw get taken by the coach's choice of substitute?", "category": "CE did not help",
         "gold": ["his coach shall designate an eligible substitute from the bench"]},
    ],
}
FAILURE = FAILURE_SETS[DOC]
print(len(FAILURE), "queries for", DOC)

7 queries for nba


In [9]:
# (a) Top-10 for every query: bi-encoder vs cross-encoder rank and score, gold chunk flagged (no LLM call)
from IPython.display import display
for item in FAILURE:
    print(f"\n=== [{item['category']}] {item['query']}")
    display(show(item["query"], item["gold"]))


=== [negation] Can a player be granted a timeout while airborne?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,107,23,1,0.611,2,7.028,"timeout shall be granted, and, upon resumption...",False
1,109,24,2,0.590,6,6.174,- 24 - k. Requests for a timeout in excess of ...,False
2,110,24,3,0.586,7,6.051,no excessive timeout will be charged and play ...,False
3,208,39,4,0.579,5,6.295,free throw and play shall resume with a throw-...,False
4,108,23,5,0.577,8,5.650,i. If a player is injured as a result of a pla...,False
5,103,23,6,0.558,4,6.597,the Official Scorer to take it and charge it t...,False
6,113,24,7,0.555,1,10.188,would be awarded possession pursuant to Rule 4...,True
7,112,24,8,0.552,3,7.013,for a timeout: (1) after the ball is released ...,False
8,345,60,9,0.551,9,4.881,"shooter, or the ball being tossed on a jump ba...",False
9,368,64,10,0.526,10,4.058,timeout (even if the Challenge is successful)....,False



=== [keyword trap] Does the shot clock reset to 14 or 24 after a foul?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,140,29,1,0.774,2,10.631,Shot Clock a. The shot clock shall be reset wh...,False
1,136,28,2,0.712,3,9.909,on the shot clock in which to attempt a field ...,False
2,308,55,3,0.708,6,8.128,of time on the shot clock. (3) For a called fo...,False
3,139,29,4,0.705,5,8.406,dead for any reason other than a defensive thr...,False
4,401,69,5,0.693,10,5.728,"when this situation occurs, the period is over...",False
5,320,57,6,0.692,8,7.095,- 57 - in action prior to any subsequent chang...,False
6,317,56,7,0.690,9,7.013,"clock should have expired, the points will not...",False
7,187,36,8,0.686,4,9.544,of interruption. The shot clock shall remain t...,False
8,261,48,9,0.679,1,10.732,- 48 - b. Double personal fouls shall add to a...,True
9,138,29,10,0.677,7,7.200,"rim, a shot clock violation has occurred. If t...",False



=== [negation] Can a coach walk past half court?


...

,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,101,22,1,0.614,2,9.387,field goal attempt. c. Officials may not use o...,False
1,100,22,2,0.604,3,7.667,IV—Tie Score—Overtime If the score is tied at ...,False
2,109,24,3,0.597,10,5.111,- 24 - k. Requests for a timeout in excess of ...,False
3,105,23,4,0.555,7,6.713,the fourth period and/or last two minutes of a...,False
4,98,22,5,0.548,9,5.748,If a field goal attempt is in flight toward th...,False
5,52,14,6,0.544,5,6.969,shall be stopped at the expiration of time for...,False
6,365,64,7,0.542,4,7.341,"team’s mandatory timeout, the team that is ult...",False
7,97,22,8,0.542,6,6.723,- 22 - e. A team is permitted a total of 30 se...,False
8,114,24,9,0.536,8,5.781,This rule also applies to the midcourt line ex...,False
9,102,23,10,0.535,1,9.587,"- 23 - b. In overtime periods, each team shall...",True



=== [buried] What makes a shot attempt legal for the shot clock?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,50,14,1,0.764,5,8.388,shot clocks/game clocks located above the back...,False
1,132,28,2,0.749,2,8.981,"position, with positions being alternated ther...",False
2,401,69,3,0.717,9,7.688,"when this situation occurs, the period is over...",False
3,136,28,4,0.716,3,8.628,on the shot clock in which to attempt a field ...,False
4,51,14,5,0.706,7,7.991,been stopped and the ball is put in play by a ...,False
5,402,69,6,0.703,4,8.412,and the officials are not reasonably certain t...,False
6,133,28,7,0.700,1,9.383,a team gains new possession of a ball which is...,True
7,139,29,8,0.698,8,7.721,dead for any reason other than a defensive thr...,False
8,400,69,9,0.692,10,6.742,be subject to appropriate penalties. L. EXPIRA...,False
9,138,29,10,0.689,6,8.016,"rim, a shot clock violation has occurred. If t...",False



=== [CE did not help] Does an injured player's free throw get taken by the coach's choice of substitute?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,254,46,1,0.789,2,10.555,of the court at the free throw line extended. ...,False
1,277,50,2,0.751,3,10.479,will be selected by his coach. The two free th...,False
2,168,33,3,0.747,1,10.784,"attempt is released, no violations can occur. ...",False
3,64,16,4,0.693,9,7.683,"an injury or ejection. (EXCEPTION: Rule 6, Sec...",False
4,256,47,5,0.691,7,9.039,will select a substitute and any player from t...,False
5,66,16,6,0.685,10,7.627,in the backcourt. If a substitute fails to mee...,False
6,273,50,7,0.681,4,10.412,- 50 - (2) If a flagrant foul—penalty (1) is a...,False
7,63,16,8,0.675,8,8.619,substitution occurs between periods or during ...,False
8,169,33,9,0.667,6,10.175,re-enter the game. The substitute must remain ...,False
9,171,34,10,0.662,5,10.364,- 34 - squad to attempt the free throw(s). The...,True


In [10]:
# (b) Run both pipelines once per query (includes the two LLM calls) and keep every result
results = {item["query"]: run(item["query"]) for item in FAILURE}
print("done:", len(results), "queries")

done: 7 queries


In [11]:
# (c) Rank-shift table: best gold-chunk rank under the bi-encoder vs under the cross-encoder
rows = []
for item in FAILURE:
    q, out = item["query"], results[item["query"]]
    gold_ids = [c["id"] for c in chunks if is_gold(c["id"], item["gold"])]
    base_rank = next((r for r, cid in enumerate(out["base_ids"], start=1) if cid in gold_ids), None)
    ce_rank = next((r for r, cid in enumerate(out["rr_ids"], start=1) if cid in gold_ids), None)
    rows.append({
        "query": q,
        "category": item["category"],
        "gold_chunks": gold_ids,
        "page": sorted({chunks[i]["page"] for i in gold_ids}),
        "base_rank": base_rank,
        "ce_rank": ce_rank,
        "moved": f"#{base_rank} \u2192 #{ce_rank}" if base_rank else "not in top-10",
        "base_top3": out["base_ids"][:TOP_N],
        "ce_top3": out["rr_ids"][:TOP_N],
    })
rank_shift = pd.DataFrame(rows)
display(rank_shift)

,query,category,gold_chunks,page,base_rank,ce_rank,moved,base_top3,ce_top3
0,Can a player be granted a timeout while airborne?,negation,[113],[24],7,1,#7 → #1,"[107, 109, 110]","[113, 107, 112]"
1,Does the shot clock reset to 14 or 24 after a ...,keyword trap,[261],[48],9,1,#9 → #1,"[140, 136, 308]","[261, 140, 136]"
2,Can a coach walk past half court?,negation,[58],[15],4,1,#4 → #1,"[59, 152, 60]","[58, 59, 152]"
3,Can a substitute replace the player shooting f...,exception,[63],[16],5,1,#5 → #1,"[168, 156, 254]","[63, 168, 256]"
4,What are the mandatory timeouts in each period?,buried,[102],[23],10,1,#10 → #1,"[101, 100, 109]","[102, 101, 100]"
5,What makes a shot attempt legal for the shot c...,buried,"[133, 134]",[28],7,1,#7 → #1,"[50, 132, 401]","[133, 132, 136]"
6,Does an injured player's free throw get taken ...,CE did not help,[171],[34],10,5,#10 → #5,"[254, 277, 168]","[168, 254, 277]"


In [12]:
# (d) Both answers per query, printed in full with the chunks each pipeline was given
def label(ids):
    return ", ".join(f"chunk {i} (p. {chunks[i]['page']})" for i in ids[:TOP_N])

for item in FAILURE:
    q, out = item["query"], results[item["query"]]
    print("=" * 100)
    print("Q:", q, f"[{item['category']}]")
    print(f"\nAnswer 1 (baseline top-{TOP_N}: {label(out['base_ids'])}):\n{out['answer1']}")
    print(f"\nAnswer 2 (re-ranked top-{TOP_N}: {label(out['rr_ids'])}):\n{out['answer2']}\n")
    # Gold chunk text (first 2 if overlap gives more) so a reader can check each answer against the rule
    for c in [c for c in chunks if is_gold(c["id"], item["gold"])][:2]:
        print(f"Gold chunk {c['id']} (p. {c['page']}): {c['text']}\n")

Q: Can a player be granted a timeout while airborne? [negation]

Answer 1 (baseline top-3: chunk 107 (p. 23), chunk 109 (p. 24), chunk 110 (p. 24)):
Not covered in the provided excerpts.

Answer 2 (re-ranked top-3: chunk 113 (p. 24), chunk 107 (p. 23), chunk 112 (p. 24)):
Based on the provided excerpts, a player shall not be granted any timeout if both of their feet are in the air and any part of their body has broken the vertical plane of the boundary line (p. 24).

Gold chunk 113 (p. 24): would be awarded possession pursuant to Rule 4, Section XIV). d. When a team is granted a timeout, play shall not resume until the Time-out Clock has expired. The throw-in shall be nearest the spot where play was suspended. The throw-in shall be on the sideline, if the ball was in play when the request was granted. e. A player shall not be granted any timeout if both of his feet are in the air and any part of his body has broken the vertical plane of the boundary line. This rule also applies to the 

## 8. Latency profiling
After one warm-up retrieve and rerank, each failure query is timed 10 times for the bi-encoder retrieval and for the cross-encoder re-rank of its top-10 (Table A). Table B uses one measurement per query taken from the section 7 runs, with no new LLM calls. All values are in milliseconds. LLM time depends on answer length, so Table A isolates the cost of re-ranking.

In [13]:
qs = [item["query"] for item in FAILURE]

# Warm-up so one-time MPS/kernel setup is not counted
_ids, _ = retrieve(qs[0]); rerank(qs[0], _ids)

REPS = 10
ret, rr = [], []   # milliseconds, one entry per (query, repetition)
for q in qs:
    for _ in range(REPS):
        t = time.perf_counter(); ids, _ = retrieve(q);  ret.append((time.perf_counter() - t) * 1000)
        t = time.perf_counter(); rerank(q, ids);        rr.append((time.perf_counter() - t) * 1000)
p2_ret = [a + b for a, b in zip(ret, rr)]   # retrieve + rerank from the same repetition

def stats(v):
    v = np.array(v)
    return {"mean_ms": v.mean(), "median_ms": np.median(v), "p95_ms": np.percentile(v, 95)}

# Table A: retrieval stages, 10 reps x every failure query
table_a = pd.DataFrame({
    "Retrieve (embed + FAISS)": stats(ret),
    "Rerank (CE on 10)": stats(rr),
    "Pipeline 1 retrieval": stats(ret),
    "Pipeline 2 retrieval (retrieve + rerank)": stats(p2_ret),
}).T.round(1)
print(f"Table A: retrieval stages (n = {len(ret)} timings)")
display(table_a)

# Table B: generation and end-to-end, one value per query from the section 7 runs
ms = lambda key: [results[q][key] * 1000 for q in qs]   # seconds to ms
llm1, llm2 = ms("t_llm1"), ms("t_llm2")
p1_total = [a + b for a, b in zip(ms("t_retrieve"), llm1)]
p2_total = [a + b + c for a, b, c in zip(ms("t_retrieve"), ms("t_rerank"), llm2)]
table_b = pd.DataFrame({
    "LLM (P1 context)": stats(llm1),
    "LLM (P2 context)": stats(llm2),
    "Pipeline 1 end-to-end (t_retrieve + t_llm1)": stats(p1_total),
    "Pipeline 2 end-to-end (t_retrieve + t_rerank + t_llm2)": stats(p2_total),
}).T.round(1)
print(f"\nTable B: generation and end-to-end (n = {len(qs)} queries)")
display(table_b)
print(f"\ndevice: {DEVICE} | bi-encoder: {EMBED_MODEL} | cross-encoder: {RERANK_MODEL} | LLM: {LLM_MODEL}")

Table A: retrieval stages (n = 70 timings)


,mean_ms,median_ms,p95_ms
Retrieve (embed + FAISS),5.8,5.5,7.6
Rerank (CE on 10),735.4,693.9,1213.0
Pipeline 1 retrieval,5.8,5.5,7.6
Pipeline 2 retrieval (retrieve + rerank),741.2,699.2,1218.0



Table B: generation and end-to-end (n = 7 queries)


,mean_ms,median_ms,p95_ms
LLM (P1 context),2127.3,1428.1,4477.9
LLM (P2 context),1181.6,1168.3,1441.2
Pipeline 1 end-to-end (t_retrieve + t_llm1),2153.3,1458.4,4506.9
Pipeline 2 end-to-end (t_retrieve + t_rerank + t_llm2),1823.0,1804.5,2114.2



device: mps | bi-encoder: sentence-transformers/all-MiniLM-L6-v2 | cross-encoder: cross-encoder/ettin-reranker-1b-v1 | LLM: gemini-3.5-flash-lite


## 9. Scaling
For one query, time the cross-encoder over all chunks versus over the top-10 candidates, and the FAISS search over the full index. The all-chunk pass is warmed up on the top-10 pairs only and timed over 2 repetitions; the top-10 pass gets one warm-up and 3 repetitions, FAISS one warm-up and 20 (medians reported). The cost of scoring every chunk is then extrapolated linearly to 1,000,000 chunks.

In [14]:
scale_q = "What happens if a defensive player is in the lane for more than three seconds?" if DOC == "nba" \
          else "Who may challenge a play after the two-minute warning?"

def median_ms(fn, reps=3, warmup=None):
    """Warm up once (with `warmup` if given, else `fn`), then return the median wall time of `reps` calls, in milliseconds."""
    (warmup or fn)()
    times = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)

N = len(chunks)
qv = bi.encode([scale_q], normalize_embeddings=True).astype("float32")
top_ids, _ = retrieve(scale_q)

top10_pairs = lambda: ce.predict([(scale_q, chunks[i]["text"]) for i in top_ids])
ce_all_ms = median_ms(lambda: ce.predict([(scale_q, c["text"]) for c in chunks]), reps=2, warmup=top10_pairs)   # every chunk
ce_10_ms  = median_ms(top10_pairs)                                                                              # top-10 only
faiss_ms  = median_ms(lambda: index.search(qv, TOP_K), reps=20)                                       # search over all N vectors

extrap_s = ce_all_ms / N * 1_000_000 / 1000   # linear extrapolation, in seconds
print(pd.DataFrame([{
    "N chunks": N,
    "CE-all ms": round(ce_all_ms, 1),
    "CE-10 ms": round(ce_10_ms, 1),
    "FAISS ms": round(faiss_ms, 3),
    "CE-all @ 1M chunks": f"{extrap_s:,.0f} s (~{extrap_s/3600:.1f} h)",
}]).to_string(index=False))

 N chunks  CE-all ms  CE-10 ms  FAISS ms  CE-all @ 1M chunks
      422    60422.6    1269.8     0.021 143,182 s (~39.8 h)


## 10. Bi-encoder comparison
This appendix compares candidate bi-encoders on retrieval only, with no cross-encoder and no LLM: each model embeds the same 120-word chunks and ranks all of them for every query in `BI_EVAL[DOC]`. The eval set is the section 7 failure set plus extra NFL probe queries whose gold snippets match at least one chunk. The failure queries were picked because the MiniLM baseline struggled on them, so this set is biased against the baseline.

In [15]:
# Extra NFL probe queries, same format as the failure set. Every gold snippet was checked with is_gold against all chunks and matches at least one.
NFL_PROBES = [
    ("What happens when the kickoff lands short of the landing zone?", "synonym trap",
     ["A kick that lands or touches a player short of the landing zone is treated like a kickoff out of bounds", "lands or is touched short of the landing zone"]),
    ("What contact is a defender allowed to make on a receiver beyond five yards?", "exception",
     ["Beyond the five-yard zone, incidental contact may exist between receiver and defender"]),
    ("Is a hip-drop tackle legal?", "negation",
     ["It is a foul if a player uses the following technique to bring a runner to the ground"]),
    ("Can the defense run into the kicker?", "negation",
     ["run into the kicker before he recovers his balance"]),
    ("How long does the offense have to snap the ball before delay of game?", "buried",
     ["It is a delay of the game if the ball is not put in play by a snap within 40 seconds"]),
    ("Can a player wear long sleeve undergarments with grip sleeves?", "negation",
     ["Players may not wear long-sleeved undergarments that include pebble-grip sleeves"]),
    ("When is the line to gain not ten yards from the snap?", "exception",
     ["except when a goal line is less than 10 yards from this spot"]),
    ("Can a personal foul after the end of a quarter be enforced on the next period?", "exception",
     ["following the end of the second or fourth periods, the penalty yardage will be enforce"]),
    ("What if the defense has too many players on the field after a substitution?", "exception",
     ["no penalties will be enforced, except for personal fouls and unsportsmanlike conduct fouls, and the down will be replayed"]),
    ("Can the quarterback spike the ball to stop the clock?", "exception",
     ["spiking or throwing the ball in the field of play after a down has ended, except after a touchdown"]),
    ("What is the penalty if a coach throws the red flag with no timeouts left?", "keyword",
     ["Loss of 15 yards enforced as a foul between downs"]),
    ("How many timeouts does each team have in overtime?", "buried",
     ["Each team is entitled to three timeouts during a half"]),
    ("Can a defender grab an opponent's facemask?", "negation",
     ["he must immediately release it. He shall not grasp and control, twist, turn, push, or pull the facem"]),
    ("Can a player be moving toward the line of scrimmage when the ball is snapped?", "negation",
     ["No player is permitted to be moving toward the line of scrimmage when the ball is snapped"]),
    ("Can a team challenge an interception ruling?", "exception",
     ["the play cannot be challenged by either team"]),
    ("Can the spot of the ball be challenged if it is closer to the goal line?", "negation",
     ["The dead ball spot is not reviewable to determine solely whether it should be closer to or further from the goal line"]),
    ("What happens in overtime when the second team kicks a field goal?", "buried",
     ["If B scores a field goal, the game continues in sudden death"]),
    ("Who may recover a fumble on fourth down?", "distractor",
     ["The player who fumbled is the only Team A player permitted to recover and advance the ball prior to a change of possession"]),
    ("Can a fourth down fumble be advanced by the defense?", "distractor",
     ["The ball may be advanced by any member of the defensive team"]),
    ("Is a player allowed to throw a helmet at an opponent?", "negation",
     ["A player may not use a helmet that is no longer worn by anyone as a weapon"]),
    ("What if a defender touches the quarterback after he spikes the ball following the two-minute warning?", "keyword",
     ["spiking or throwing the ball in the field of play after a down has ended, except after a touchdown"]),
    ("Can the offense hurry to snap the ball while a substitution is in progress?", "negation",
     ["the offense is prohibited from rushing quickly to the line of scrimmage and snapping the ball in an obvious attempt"]),
    ("Are tear-away jerseys allowed?", "negation",
     ["Tear-away jerseys are prohibited"]),
    ("Can players roll their pants up above the knee?", "negation",
     ["pants shortened or rolled up are prohibited"]),
    ("Can team staff walk on the field during halftime?", "negation",
     ["Representatives of either team are prohibited from entering the field, unless they are incoming substitutes"]),
    ("Can the head coach go on the field during a game?", "negation",
     ["any game-type activities are prohibited on the field of play"]),
    ("Can a player wear transparent shields on his helmet visor?", "negation",
     ["transparent materials are prohibited"]),
    ("Who loses possession first in overtime if the first team kicks a field goal and the second team fumbles?", "buried",
     ["If the first team to possess the ball scores a field goal or touchdown, after which the second team loses possession by an interception or fumble"]),
]

BI_EVAL = {
    "nfl": FAILURE_SETS["nfl"] + [{"query": q, "category": c, "gold": g} for q, c, g in NFL_PROBES],
    "nba": FAILURE_SETS["nba"],
}
eval_set = BI_EVAL[DOC]
eval_queries = [item["query"] for item in eval_set]
eval_gold = [{c["id"] for c in chunks if is_gold(c["id"], item["gold"])} for item in eval_set]
assert all(eval_gold), "every eval query needs at least one gold chunk"

BGE_PREFIX = "Represent this sentence for searching relevant passages: "
# name -> (HF id, query prefix, document prefix, trust_remote_code)
BI_CANDIDATES = {
    "all-MiniLM-L6-v2 (baseline)": ("sentence-transformers/all-MiniLM-L6-v2", "", "", False),
    "multi-qa-MiniLM-L6-cos-v1":   ("sentence-transformers/multi-qa-MiniLM-L6-cos-v1", "", "", False),
    "bge-small-en-v1.5":           ("BAAI/bge-small-en-v1.5", BGE_PREFIX, "", False),
    "bge-base-en-v1.5":            ("BAAI/bge-base-en-v1.5", BGE_PREFIX, "", False),
    "nomic-embed-text-v1.5":       ("nomic-ai/nomic-embed-text-v1.5", "search_query: ", "search_document: ", True),
}

def _extended_attention_mask(self, attention_mask, input_shape=None, device=None, dtype=None):
    """transformers v4 helper that v5 removed; nomic's remote code still calls it.
    Returns 0 for real tokens and a large negative number for padding."""
    dtype = dtype or self.dtype
    return (1.0 - attention_mask[:, None, None, :].to(dtype)) * torch.finfo(dtype).min


def load_bi_encoder(hf_id, trust):
    model = SentenceTransformer(hf_id, device=DEVICE, trust_remote_code=trust)
    inner = model[0].auto_model
    if trust and not hasattr(inner, "get_extended_attention_mask"):
        type(inner).get_extended_attention_mask = _extended_attention_mask
    return model


chunk_texts = [c["text"] for c in chunks]   # the shared chunks; a document prefix is added only while encoding
rank_cols, summary = {}, []
for name, (hf_id, q_prefix, d_prefix, trust) in BI_CANDIDATES.items():
    try:
        model = load_bi_encoder(hf_id, trust)
    except Exception as e:   # one broken model should not stop the rest of the comparison
        print(f"skipped {name}: {type(e).__name__}: {e}")
        continue
    model.encode(chunk_texts[:8], normalize_embeddings=True)   # warm-up so one-time device setup is not timed
    t = time.perf_counter()
    doc_emb = model.encode([d_prefix + x for x in chunk_texts], normalize_embeddings=True, batch_size=64)
    encode_s = time.perf_counter() - t
    q_emb = model.encode([q_prefix + q for q in eval_queries], normalize_embeddings=True)
    order = np.argsort(-(q_emb @ doc_emb.T), axis=1)   # every chunk ranked by dot product, best first
    best = np.array([int(np.isin(row, list(g)).argmax()) + 1 for row, g in zip(order, eval_gold)])   # best rank of any gold chunk
    rank_cols[name] = best
    summary.append({"model": name, "recall@3": (best <= 3).mean(), "recall@10": (best <= 10).mean(),
                    "MRR@10": np.where(best <= 10, 1 / best, 0).mean(), "encode s": encode_s})
    del model
    if DEVICE == "mps":
        torch.mps.empty_cache()

print(f"(a) Best gold-chunk rank among all {len(chunks)} chunks ({len(eval_queries)} queries, {DOC})")
display(pd.DataFrame(rank_cols, index=eval_queries))
print("\n(b) Summary per model")
display(pd.DataFrame(summary).set_index("model").round({"recall@3": 3, "recall@10": 3, "MRR@10": 3, "encode s": 1}))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14250.34it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13326.75it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

<All keys matched successfully>


(a) Best gold-chunk rank among all 422 chunks (7 queries, nba)


,all-MiniLM-L6-v2 (baseline),multi-qa-MiniLM-L6-cos-v1,bge-small-en-v1.5,bge-base-en-v1.5,nomic-embed-text-v1.5
Can a player be granted a timeout while airborne?,7,3,12,3,1
Does the shot clock reset to 14 or 24 after a foul?,9,17,10,3,12
Can a coach walk past half court?,4,2,8,2,1
Can a substitute replace the player shooting free throws?,5,8,6,2,1
What are the mandatory timeouts in each period?,10,6,2,2,1
What makes a shot attempt legal for the shot clock?,7,4,4,2,1
Does an injured player's free throw get taken by the coach's choice of substitute?,10,23,6,16,2



(b) Summary per model


,recall@3,recall@10,MRR@10,encode s
model,,,,
all-MiniLM-L6-v2 (baseline),0.000,1.000,0.150,0.4
multi-qa-MiniLM-L6-cos-v1,0.286,0.714,0.196,0.5
bge-small-en-v1.5,0.143,0.857,0.187,0.8
bge-base-en-v1.5,0.857,0.857,0.381,2.6
nomic-embed-text-v1.5,0.857,0.857,0.786,5.5
